# LangChain: Models, Messages & Structured Output

## Outline
* Installation and setup
* Comparison of LangChain with direct LiteLLM / OpenAI / Ollama usage
* init_chat_model — Connecting to any provider
* Messages — Message types
* Prompt Templates
* Streaming
* Structured Output with Pydantic
* Token Usage


## 1. Installation

In [ ]:
# Install the required packages
# pip install -U langchain langchain-openai langchain-ollama python-dotenv


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

## 2. Why LangChain?

- **LangChain**: In addition to provider integration, it offers complete tools for building agents, memory, RAG, and evaluation


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.2", model_provider="openai",
                        api_key=api_key,
                        base_url=base_url,
                        temperature=0)
response = model.invoke("Hello! What can you do?")
print(response.content)


I can help with a wide range of tasks—here are the main things:

- **Answer questions & explain concepts** (science, history, math, programming, finance basics, etc.)
- **Write and edit** emails, resumes, reports, essays, blog posts; improve clarity, tone, and grammar
- **Summarize and extract key points** from text (and from images you share)
- **Brainstorm ideas** for projects, names, content, lesson plans, research topics
- **Plan and organize** schedules, study plans, workout plans, travel itineraries, checklists
- **Programming help**: debug code, explain errors, write snippets, design algorithms, review code
- **Data tasks**: help design tables, interpret results, create formulas (Excel/Sheets), basic analysis guidance
- **Image help** (if you upload one): describe what’s in it, read text, compare images, troubleshoot diagrams
- **Role-play & practice**: interviews, conversations, presentations, negotiation prep

If you tell me what you’re working on (and your goal + any constrai

## 3. init_chat_model — Connecting to any Provider

The most important change compared with older LangChain versions: you no longer need a separate import for each provider.


In [18]:
from langchain.chat_models import init_chat_model

# Ollama
#model_ollama = init_chat_model("gemma3:4b", model_provider="ollama", temperature=0)

# Anthropic (requires: pip install langchain-anthropic)
model_anthropic = init_chat_model("claude-sonnet-4-6", model_provider="openai", api_key=api_key, base_url=base_url)
#model_anthropic = init_chat_model("claude-sonnet-4-6", api_key=api_key, base_url=base_url)

# Ollama — local model (requires: pip install langchain-ollama)
# model_ollama = init_chat_model("llama3.2", model_provider="ollama")

# OpenAI-compatible (such as a LiteLLM proxy or vLLM)
# model_custom = init_chat_model(
#     model="your-model",
#     model_provider="openai",
#     base_url="http://localhost:11434/v1",
#     api_key="dummy"
# )

print(type(model_anthropic))


<class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [19]:
a = model_anthropic.invoke("Hello, how are you? What is your name?")
a.content

"Hello! I'm doing well, thank you for asking! I'm **Claude**, an AI assistant made by Anthropic. How are you doing today? Is there something I can help you with?"

In [45]:
# Important parameters
model = init_chat_model(
    "gpt-5.2",
    model_provider="openai",
    temperature=0.7,       # creativity (0 to 2)
    max_tokens=500,        # maximum output tokens
    timeout=30,            # timeout in seconds
    max_retries=3,         # number of retries in case of an error
    api_key=api_key, 
    base_url=base_url
)

In [42]:
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "llama3:latest",
        "messages": [
            {"role": "user", "content": "Hello! What is your name"}
        ],
        "stream": False
    }
)

print(response.status_code)
print(response.text)

200
{"model":"llama3:latest","created_at":"2026-08-14T11:33:13.0376964Z","message":{"role":"assistant","content":"Nice to meet you! I don't have a personal name, as I'm just an AI designed to assist and communicate with humans. You can call me \"Assistant\" or simply \"AI\" if you like. I'm here to help answer questions, provide information, and engage in conversation - so feel free to get started!"},"done":true,"done_reason":"stop","total_duration":4359243500,"load_duration":3849265300,"prompt_eval_count":16,"prompt_eval_duration":45099000,"eval_count":67,"eval_duration":456552000}


In [ ]:
model_ollama = init_chat_model("llama3:latest", model_provider="ollama", temperature=0)
#model_ollama = init_chat_model("gpt-oss:20b", model_provider="ollama", temperature=0)
a = model_ollama.invoke("Hello, how are you? What is your name?")
a.content

In [36]:
!ollama list

NAME             ID              SIZE      MODIFIED      
kimi-k3:cloud    e8aa77394b8b    -         13 days ago      
gpt-oss:120b     a951a23b46a1    65 GB     6 months ago     
llama3:latest    365c0bd3c000    4.7 GB    7 months ago     
gpt-oss:20b      aa4295ac10c3    13 GB     11 months ago    


## 4. Messages — Message types

In LangChain, everything works with messages. This maps directly to the OpenAI API.


In [46]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# Three main message types
system = SystemMessage("You are a helpful Persian-speaking assistant.")
human = HumanMessage("Hello! My name is Meisam.")
ai = AIMessage("Hello Meisam! How can I help you?")

# Send the full conversation
conversation = [system, human, ai, HumanMessage("What is my name?")]
response = model.invoke(conversation)
print(response.content)
print(f"\nResponse type: {type(response)}")

نام شما **میثم** است.

Response type: <class 'langchain_core.messages.ai.AIMessage'>


In [47]:
# You can also use dictionaries (like the direct OpenAI API)
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]
response = model.invoke(messages)
print(response.content)


2 + 2 = 4


In [52]:
print(response)

content='Why don’t skeletons fight each other?\n\nThey don’t have the guts.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 10, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 7, 'engine_ttft_ms': 47, 'engine_ttlt_ms': 187, 'pre_inference_ms': 81, 'service_tbt_ms': 7, 'service_ttft_ms': 534, 'service_ttlt_ms': 669, 'total_duration_ms': 596, 'user_visible_ttft_ms': 454}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-ECkVMpDJqa8lE7Nniirze8HGobGbF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0000f-334e-7712-90af-4872ddf2753d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'

In [51]:
# Metadata and token usage in the response
response = model.invoke("Tell me a joke")
print(f"Content: {response.content}")
print(f"Model: {response.response_metadata.get('model_name', 'N/A')}")
print(f"Usage: {response.usage_metadata}")


Content: Why don’t skeletons fight each other?

They don’t have the guts.
Model: gpt-5.2-2025-12-11
Usage: {'input_tokens': 10, 'output_tokens': 20, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 5. Prompt Templates

A prompt template is like a function that takes input and returns a prepared prompt.


In [54]:
from langchain_core.prompts import ChatPromptTemplate

# Simplest form
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that translates {input_language} to {output_language}.
     You must only translate the text with no description, no explanation, no additional text."""),
    ("human", "{text}")
])

# Fill the template
messages = prompt.invoke({
    "input_language": "English",
    "output_language": "Persian",
    "text": "Hello, how are you? My name is X"
})
print(messages)


messages=[SystemMessage(content='You are a helpful assistant that translates English to Persian.\n     You must only translate the text with no description, no explanation, no additional text.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you? My name is X', additional_kwargs={}, response_metadata={})]


In [56]:
response = model.invoke(messages)
print(response.content)

سلام، حال شما چطور است؟ نام من X است


In [57]:
messages = prompt.invoke({
    "input_language": "Persian",
    "output_language": "Arabic",
    "text": "What is your name and how old are you?"
})
response = model.invoke(messages)
print(response.content)

ما اسمك وكم عمرك؟


## 5 . LCEL: LangChain Expression Language

**Building a `chain` with the `|` operator in LCEL:** Connects the prompt and model, then invokes the chain with a dictionary of prompt variables and prints the model's final output.

In [58]:
# Use in a chain with LCEL (pipe operator)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that translates {input_language} to {output_language}.
     You must only translate the text with no description, no explanation, no additional text."""),
    ("human", "{text}")
])

chain = prompt | model

response = chain.invoke({
    "input_language": "English",
    "output_language": "Persian",
    "text": "LangChain is a powerful framework for building LLM applications."
})
print(response.content)


لنگ‌چین یک چارچوب قدرتمند برای ساخت برنامه‌های کاربردی مبتنی بر مدل‌های زبانی بزرگ (LLM) است.


In [59]:
# Few-shot prompt — giving examples to the model
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sentiment analyzer. Reply with only: POSITIVE, NEGATIVE, or NEUTRAL"),
    ("human", "I love this product!"),
    ("ai", "POSITIVE"),
    ("human", "This is terrible."),
    ("ai", "NEGATIVE"),
    ("human", "It's okay, nothing special."),
    ("ai", "NEUTRAL"),
    ("human", "{text}"),
])

chain = few_shot_prompt | model
result = chain.invoke({"text": "Best purchase I've made this year!"})
print(result.content)


POSITIVE


In [60]:
result = chain.invoke("Best purchase I've made this year!")
print(result.content)

POSITIVE


## 6. Streaming

With streaming, you can display tokens as they are generated — like ChatGPT.


In [65]:
# Simple streaming
print("Streaming output:")
for chunk in model.stream("Write a blog post about agent usage in partile physics. Your output should be 3 paragraphs long"):
    print(chunk.content, end="", flush=True)
print()  # newline


Streaming output:
Agent-based approaches are becoming a practical way to accelerate work in particle physics, not by replacing theory or experiment, but by helping researchers manage complexity across massive datasets, intricate simulations, and distributed collaborations. In this context, an “agent” is typically a software system that can take goals (for example, “validate this detector calibration” or “scan this parameter space”) and then plan, execute, and iterate: pulling the right data, running tools, checking constraints, and summarizing results. Particle physics is a natural fit because modern analyses often involve long chains of steps—data quality checks, event reconstruction, feature engineering, statistical inference, and documentation—where mistakes can be subtle and provenance matters. Agents can act as reliable “process glue,” making workflows more reproducible while reducing the manual overhead that can slow down discovery.

One of the clearest areas of impact is simulat

In [66]:
# batch — send multiple requests in parallel
responses = model.batch([
    "What is Python?",
    "What is JavaScript?",
    "What is Rust?",
])
for i, r in enumerate(responses):
    print(f"Response {i+1}: {r.content[:80]}...")

Response 1: Python is a high-level, general-purpose programming language designed to be easy...
Response 2: JavaScript is a high-level programming language used to make websites interactiv...
Response 3: Rust is a modern programming language designed for building fast, reliable softw...


## 7. Structured Output with Pydantic

Instead of parsing text, force the model to directly output structured data.
This replaces the legacy `ResponseSchema` and `StructuredOutputParser`.

In [69]:
from pydantic import BaseModel, Field
from typing import List

# Define the schema with Pydantic
class ProductReview(BaseModel):
    """Information extracted from a product review"""
    gift: bool = Field(description="Was the product purchased as a gift?")
    delivery_days: int = Field(description="How many days did delivery take? Use -1 if unavailable.")
    #price_value: List[str] = Field(description="Statements about the product's price or value")
    price_value: List[str] = Field(description="price for objects")

# Create a model with structured output
structured_model = model.with_structured_output(ProductReview)

In [70]:
review = """
This leaf blower is pretty amazing. It has four settings: candle blower, gentle breeze, 
windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. 
I think my wife liked it so much she was speechless. I paid 35 dollars for this for the tornado 
setting alone but you can get it for about half the price!
"""

result = structured_model.invoke(f"Extract info from this review: {review}")
print(f"Gift: {result.gift}")
print(f"Delivery days: {result.delivery_days}")
print(f"Price comments: {result.price_value}")
print(f"\nType: {type(result)}")


Gift: True
Delivery days: 2
Price comments: ['35 dollars', 'about half the price']

Type: <class '__main__.ProductReview'>


In [ ]:
# Another example: extracting structured information
class MovieInfo(BaseModel):
    """Movie information"""
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    director: str = Field(description="Director")
    rating: float = Field(description="Rating out of 10")

movie_model = model.with_structured_output(MovieInfo)
try:
    result = movie_model.invoke("Tell me about the movie Inception")
except:
    print("I cant follow your structure.")
    result = None
print(result)
print(f"\nTitle: {result.title}, Year: {result.year}, Director: {result.director}")


title='Inception' year=2010 director='Christopher Nolan' rating=8.8

Title: Inception, Year: 2010, Director: Christopher Nolan


## 8. Token Usage — Tracking usage

This is important for controlling costs.


In [74]:
from langchain_core.callbacks import get_usage_metadata_callback

#model_gpt = init_chat_model("gpt-4o", model_provider="openai")

with get_usage_metadata_callback() as cb:
    model.invoke("Hello!")
    model.invoke("What is the capital of France?")

print(cb.usage_metadata)

{'gpt-5.2-2025-12-11': {'output_tokens': 20, 'input_token_details': {'cache_read': 0, 'audio': 0}, 'output_token_details': {'reasoning': 0, 'audio': 0}, 'total_tokens': 41, 'input_tokens': 21}}


In [75]:
# Usage is also available in each response
response = model.invoke("Explain machine learning in one sentence")
print(f"Input tokens:  {response.usage_metadata['input_tokens']}")
print(f"Output tokens: {response.usage_metadata['output_tokens']}")
print(f"Total tokens:  {response.usage_metadata['total_tokens']}")


Input tokens:  12
Output tokens: 30
Total tokens:  42
